# Lewisham · `appeal_decision` and `appeal_decision_date` (web: "Appeal decision", "Appeal decision date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `appeal_decision` and `appeal_decision_date` for every Lewisham application (paged
with `search_after`, sorted by `id`, because there are more than 10,000). Count for
`appeal_decision`:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

Then count the four combinations of the two fields being filled or not.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "appeal_decision", "appeal_decision_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "appeal_decision", "appeal_decision_date"])
print("Lewisham applications:", len(df))

Lewisham applications: 29021


In [3]:
value = df["appeal_decision"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,28273,97.42
empty or blank,0,0.00
filled,748,2.58
total,29021,100.00


In [4]:
has_decision = df["appeal_decision"].notna() & (df["appeal_decision"].astype(str).str.strip() != "")
has_date = df["appeal_decision_date"].notna() & (df["appeal_decision_date"].astype(str).str.strip() != "")
pd.crosstab(has_decision.rename("has appeal_decision"), has_date.rename("has appeal_decision_date"), margins=True)

has appeal_decision_date,False,True,All
has appeal_decision,,,
False,28273,0,28273
True,0,748,748
All,28273,748,29021


### Findings · Step 1

- **748 records (2.58%) have an appeal decision;** 28,273 have none. No empty or blank
  strings.
- **The two fields always come together:** every record with an appeal_decision has an
  appeal_decision_date, and none has a date without a decision. The same holds in Brent; its
  one exception, Brent-20_4264, comes from our portal fill, not from the Datahub.
- Null is normal here: most applications are never appealed.

## Step 2 · Type check, both fields

Aggregations hide the JSON type, so the Python type of every value in `df` (Step 1) is
counted: `str` = JSON string, `int` / `float` = JSON number, `NoneType` = null. For
`appeal_decision_date`, the text is also checked against the format `dd/MM/yyyy`.

In [5]:
def python_type(v: object) -> str:
    return "NoneType" if v is None or v != v else type(v).__name__


for field in ["appeal_decision", "appeal_decision_date"]:
    print(field, df[field].map(python_type).value_counts().to_dict())

dates = df["appeal_decision_date"].dropna()
fits = dates.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
parsed = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
print("dates in dd/MM/yyyy:", int(fits.sum()), "of", len(dates), "| not parseable:", int(parsed.isna().sum()))
print("earliest:", parsed.min().date(), "| latest:", parsed.max().date())

appeal_decision {'NoneType': 28273, 'str': 748}
appeal_decision_date {'NoneType': 28273, 'str': 748}
dates in dd/MM/yyyy: 748 of 748 | not parseable: 0
earliest: 1987-01-27 | latest: 2026-11-05


## Step 3 · Most common values, both fields

`appeal_decision`: every value against the schema's list (`shown_as` uses `repr()`).
`appeal_decision_date`: appeals by year, and the most common single dates.

### Findings · Step 2

- **Both fields are strings or null only** (748 strings each, 28,273 null). No numbers.
- **Every appeal_decision_date is `dd/MM/yyyy`** and parses as a real date (748 of 748),
  from 1987-01-27 to 2026-11-05. No future dates.

In [6]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "appeal_decision" (extracted 2026-11-10).
SCHEMA_APPEAL_DECISION = ["Allowed", "Dismissed", "Appeal In Progress", "Quashed", "Split Decision", "Withdrawn", "Appeal Closed"]

counts = df["appeal_decision"].value_counts().to_frame("count").reset_index()
counts["shown_as"] = counts["appeal_decision"].map(repr)
counts["in_schema"] = counts["appeal_decision"].isin(SCHEMA_APPEAL_DECISION)
print("schema values never seen:", [v for v in SCHEMA_APPEAL_DECISION if v not in set(df["appeal_decision"].dropna())])
counts

schema values never seen: ['Appeal In Progress', 'Quashed', 'Appeal Closed']


,appeal_decision,count,shown_as,in_schema
0,Dismissed,446,'Dismissed',True
1,Allowed,275,'Allowed',True
2,Withdrawn,16,'Withdrawn',True
3,Split Decision,11,'Split Decision',True


In [7]:
print("Appeals by year of appeal_decision_date:")
print(parsed.dt.year.value_counts().sort_index().to_string())
print()
print("Most common dates:")
print(dates.value_counts().head(10).to_string())

Appeals by year of appeal_decision_date:
appeal_decision_date
1987      3
1988      3
1989      8
1990      4
1991      1
1992      2
1994      1
1995      1
1998      1
2000      2
2001      3
2002      2
2003      2
2004      2
2005      3
2006      3
2007      9
2008     11
2009     10
2010      3
2011      4
2012      6
2013      6
2014      4
2015      7
2016      5
2018      9
2019     43
2020     82
2021    114
2022    115
2023     98
2024     77
2025     62
2026     42

Most common dates:
appeal_decision_date
01/03/2021    9
29/11/2022    6
05/02/2020    5
12/03/2021    5
16/12/2020    5
13/12/2022    5
10/12/2019    4
06/11/2020    4
25/06/2021    4
29/09/2021    4


### Findings · Step 3

| `appeal_decision` | Count | In schema |
|---|---|---|
| Dismissed | 446 | yes |
| Allowed | 275 | yes |
| Withdrawn | 16 | yes |
| Split Decision | 11 | yes |

- **Clean:** all 4 values are schema values; no case or whitespace variants, no ZZZ (Brent
  had 16).
- **Never seen:** Appeal In Progress, Quashed, Appeal Closed. As in Brent, running appeals
  show only in status (Appeal Received 30, Appeal in Progress 1).
- **More than status shows:** appeal_decision has Allowed 275, status only 140; Dismissed is
  446 in both. Same pattern as Brent (status moves on after an allowed appeal). Checked
  record by record in the next step.
- The counts match the status groups seen before: Withdrawn 16 and Split Decision 11
  (decision notebook, Steps 5–6); not yet checked record by record.
- **Dates:** 115 appeals before 2019 (1987–2018), then 633 from 2019 on, 42–115 a
  year (most in 2021–2022). No years 1993, 1996, 1997, 1999 or 2017. The most common single date has 9 appeals
  (01/03/2021): no placeholder date.

## Step 4 · `status` against `appeal_decision`

Fetch `status` for every record and count it against `appeal_decision`, to see what the
status says now for each appeal outcome, and whether any status says Allowed or Dismissed
without a matching appeal decision. Nulls are shown as `(none)`.

In [8]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "status", "appeal_decision"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

sa = pd.DataFrame(rows).reindex(columns=["id", "status", "appeal_decision"]).fillna("(none)")

# Only records that have an appeal decision, or a status that is an appeal outcome.
APPEAL_STATUS = ["Allowed", "Dismissed", "Split Decision", "Appeal Received", "Appeal In Progress", "Appeal in Progress", "ZZZ"]
subset = sa[(sa["appeal_decision"] != "(none)") | sa["status"].isin(APPEAL_STATUS)]
pd.crosstab(subset["status"], subset["appeal_decision"], margins=True)

appeal_decision,(none),Allowed,Dismissed,Split Decision,Withdrawn,All
status,,,,,,
Allowed,0,140,0,0,0,140
Appeal Received,30,0,0,0,0,30
Appeal in Progress,1,0,0,0,0,1
Commenced,0,2,0,0,0,2
Completed,0,77,0,0,0,77
Dismissed,0,0,446,0,0,446
Lapsed,0,33,0,0,0,33
Split Decision,0,0,0,11,0,11
Superseded,0,23,0,0,0,23


### Findings · Step 4

Live API, 2026-11-11, what `status` says now for each `appeal_decision`:

| `appeal_decision` | `status` now |
|---|---|
| Allowed (275) | Allowed 140, Completed 77, Lapsed 33, Superseded 23, Commenced 2 |
| Dismissed (446) | Dismissed 446 |
| Split Decision (11) | Split Decision 11 |
| Withdrawn (16) | Withdrawn 16 |
| none | Appeal Received 30, Appeal in Progress 1 |

- **No contradictions,** as in Brent: every status Allowed / Dismissed / Split Decision /
  Withdrawn-after-appeal has the same appeal_decision.
- **Confirmed record by record:** the 11 Split Decision and the 16 Withdrawn are the same
  records as in the decision notebook (Steps 5–6).
- **Status moved on after an allowed appeal:** 135 of 275 (49%) are now Completed, Lapsed,
  Superseded or Commenced. A dismissed appeal never moved on.
- **Withdrawn appeals differ from Brent:** in Lewisham all 16 keep status Withdrawn; in
  Brent 30 of 43 did, 11 show Refused, 2 Closed.
- **Running appeals have no appeal_decision yet:** Appeal Received 30, Appeal in Progress
  1 (status case variant, fixed at load).
- No status Closed after an appeal (Brent had 21).